# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [100]:
# importar librerías
import pandas as pd
import numpy as np

In [101]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [102]:
# explorar dataset orders
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


### Dataset orders.
- Contiene 25.100 registros.
- Columnas con todos los registros: id_pedido, id_usuario, fecha_hora_pedido, monto_total
- Columnas con valores faltantes: pais, dispositivo, fuente_referencia, nombre_producto, categoría_producto, cantidad, precio_unitario, monto_descuento
- Columna fecha_hora_pedido debe transformarse a formato datetime.
- Columnas categóricas: pais, dispositivo, fuente_referencia, nombre_producto, categoria_producto
- Columnas numéricas: cantidad, precio_unitario, monto_descuento, monto_total

In [103]:
#visualizamos las primeras filas de dataset orders
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [104]:
#exploramos medidas estadísticas en columnas numéricas de dataset orders
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


### Dataset orders
- Columna cantidad con outliers (negativos, max muy por sobre el percentil 75%)
- Columna monto_total con outliers (max de 8 millones muy por sobre el percentil 75% de 518)

In [105]:
#explorar variables categóricas y como se distribuyen de dataset orders

vars_categoricas_orders =['pais','dispositivo','fuente_referencia','nombre_producto','categoria_producto',]
for var in vars_categoricas_orders:
    print(f"\n=== Distribución de {var} ===")
    print(orders[var].value_counts())


=== Distribución de pais ===
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
Name: pais, dtype: int64

=== Distribución de dispositivo ===
desktop    12759
mobile     12321
Name: dispositivo, dtype: int64

=== Distribución de fuente_referencia ===
social         8428
organic        8329
paid_search    8313
Name: fuente_referencia, dtype: int64

=== Distribución de nombre_producto ===
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
Name: nombre_producto, dtype: int64

=== Distribución de categoria_producto ===
Hogar          8385
Moda           8323
Electronica    8312
Name: categoria_producto, dtype: int64


### Dataset orders
- Columna país se debe estandarizar los nombres.

In [106]:
#explorar dataset catalog
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


### Dataset catalog
- Contiene 7 registros
- No existen valores vacios
- Columnas categóricas: nombre producto, categoria_producto, proveedor
- Columna numérica: costo_unitario

In [107]:
# mostrar las siete filas dataset catalog
catalog.head(7)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [108]:
#exploramos medidas estadísticas en columnas numéricas de dataset catalog
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


### Dataset catalog
- Columna costo_unitario no contiene outliers

In [109]:
#exploramos variables categóricas y como se distribuyen en dataset catalog
vars_categoricas_catalog =['nombre_producto','categoria_producto','proveedor']
for var in vars_categoricas_catalog:
    print(f"\n=== Distribución de {var} ===")
    print(catalog[var].value_counts())



=== Distribución de nombre_producto ===
Laptop-Gaming-16GB      1
Vacuum-Pro-Black        1
Sneakers-Urban-42       1
Jacket-Winter-M         1
Blender-XL-Red          1
Phone-Pro-128GB         1
Tablet-Standard-64GB    1
Name: nombre_producto, dtype: int64

=== Distribución de categoria_producto ===
Electrónica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64

=== Distribución de proveedor ===
Long-Reid                  1
Bowers LLC                 1
Fuller, Pena and Myers     1
King Ltd                   1
Rivera, Carr and Finley    1
Greene-Smith               1
Mcmillan-Rhodes            1
Name: proveedor, dtype: int64


Categoria Electrónica en dataset catalog está con acento, y en dataset orders está sin acento (Electronica). Se estandarizará el nombre a Electronica

In [110]:
# explorar dataset marketing
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


### Dataset marketing
- Contiene 1.620 registros
- Columna fecha se debe transformar a datetime
- Columnas categóricas: pais, id_campaña, canal
- Columnas numéricas: gasto
- Contiene valores vacíos en columna canal
- Se debe renombrar columnas fecha_hora_pedido y fuente_referencia en dataset orders por fecha y canal para estandarizar.

In [111]:
# mostrar primeras cinco filas dataset marketing
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [112]:
#exploramos medidas estadísticas en columnas numéricas de dataset marketing
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


Columna gasto no contiene outliers (media similar a la mediana)

In [113]:
#exploramos variables categóricas y como se distribuyen en dataset marketing
vars_categoricas_marketing =['pais','canal']
for var in vars_categoricas_marketing:
    print(f"\n=== Distribución de {var} ===")
    print(marketing[var].value_counts())


=== Distribución de pais ===
Mexico       540
Argentina    540
Colombia     540
Name: pais, dtype: int64

=== Distribución de canal ===
paid_search    507
social         506
organic        506
Name: canal, dtype: int64


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.


Se revisan los 3 datasets
- Creamos una copia de los tres dataset para trabajar.
- Validar y convertir fechas al formato correcto.
- Estandarizamos nombres de columnas. 
- Eliminar duplicados.
- Revisar variables numéricas (negativos, inválidos y outliers)  
- Verificar consistencia de montos.  
- Revisar variables categóricas.
- Convertimos a número columnas numéricas.
- Valores nulos (decidir si eliminar o imputar)

In [114]:
# creamos copia de los datasets
orders_work = orders.copy()
catalog_work = catalog.copy()
marketing_work = marketing.copy()

In [115]:
# Como podemos ver, hay fechas en formato object en dataset orders
# validar y convertir fechas a formato datetime dataset orders
# errors='coerce' para convertir fechas inválidas en NaT
orders_work['fecha_hora_pedido']=pd.to_datetime(orders_work['fecha_hora_pedido'],errors='coerce')

In [116]:
# Verificamos el cambio
orders_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


In [117]:
# Como podemos ver, hay fechas en formato object en dataset marketing
# validar y convertir fechas a formato datetime dataset marketing
# errors='coerce' para convertir fechas inválidas en NaT
marketing_work['fecha']=pd.to_datetime(marketing_work['fecha'],errors='coerce')

In [118]:
# verificamos el cambio
marketing_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [119]:
# cambiamos nombre de columnas orders_work para estandarizarlas
orders_work = orders_work.rename(columns={'fecha_hora_pedido': 'fecha','fuente_referencia':'canal'})

In [120]:
# Verificamos el cambio
orders_work.columns

Index(['id_pedido', 'id_usuario', 'fecha', 'pais', 'dispositivo', 'canal',
       'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario',
       'monto_descuento', 'monto_total'],
      dtype='object')

In [121]:
#verificamos si existen duplicados en dataset orders (True/False por fila)
orders_work.duplicated().sum()

100

In [122]:
# eliminamos duplicados manteniendo el primero
orders_work = orders_work.drop_duplicates(keep='first')

In [123]:
# verificamos duplicados antes y despues
print(f"Filas antes: {len(orders)}")
print(f"Filas después: {len(orders_work)}")

Filas antes: 25100
Filas después: 25000


In [124]:
# verificamos si existen filas con el mismo id_pedido

duplicados_id_pedido = orders_work[orders_work['id_pedido'].duplicated(keep=False)]
print(f"Cantidad de filas con id_pedido duplicado: {len(duplicados_id_pedido)}")

Cantidad de filas con id_pedido duplicado: 0


In [125]:
# como podemos observar en orders.info(), tenemos valores fuera de rangos normales (valores negativos y un valor máximo de 20.000)
# visualizamos frecuencia de cantidad vendida
orders_work['cantidad'].value_counts()

 2.0        12591
 1.0        12345
 10000.0        6
 20000.0        4
-1.0            3
-2.0            1
Name: cantidad, dtype: int64

In [126]:
# revisamos primero las filas que tienen valores de cantidad negativos
orders_work[orders_work['cantidad'] < 0] 


,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


In [127]:
# En este caso podemos ver que el valor de la columna cantidad y monto_total están negativos. 
# Podría ser una devolución, pero no lo sabemos. 
# Filtramos el dataset para trabajar sólo con las ventas (incluyendo los valores NaN de cantidad). 
orders_work = orders_work[(orders_work['cantidad'] > 0) | (orders_work['cantidad'].isna())]

In [128]:
# Verificamos el cambio
orders_work['cantidad'].value_counts()

2.0        12591
1.0        12345
10000.0        6
20000.0        4
Name: cantidad, dtype: int64

In [129]:
# ahora revisamos las filas que tienen valores de cantidad > 999
orders_work[orders_work['cantidad'] > 999] 

,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.0
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.0
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.0
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.0
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.0
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.0
3689,order_3689,user_6566,2025-06-16,mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.0
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.0
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.0
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.0


In [130]:
# Vemos la distribución de los valores en cantidad y monto_total
orders_work[['cantidad', 'monto_total']].describe()

,cantidad,monto_total
count,24946.000000,2.499600e+04
mean,7.116452,2.079877e+03
std,296.893743,9.915553e+04
min,1.000000,5.240000e+00
25%,1.000000,1.807725e+02
50%,2.000000,3.419250e+02
75%,2.000000,5.185950e+02
max,20000.000000,8.840200e+06


- En este caso vemos que los diez valores máximos de cantidad y monto_total están muy por encima de los máximos.
- Vamos a reemplazar de forma correcta en estas filas la cantidad y monto_total winsorizando, esto es, reemplazando cualquier valor por encima del
  percentil 99 con el valor del percentil 99
- La winsorización limita la influencia de valores extremos reemplazándolos por valores percentiles cercanos, en lugar de eliminarlos.

In [131]:
# calculamos los percentiles
p99_ca = orders_work['cantidad'].quantile(0.99)
p99_mt = orders_work['monto_total'].quantile(0.99)

In [132]:

# reemplazamos valores sobre el percentil 99 con percentil 99
orders_work['cantidad'] = orders_work['cantidad'].clip(upper=p99_ca)
orders_work['monto_total'] = orders_work['monto_total'].clip(upper=p99_mt)


In [133]:
# Verificamos la imputación
orders_work[['cantidad', 'monto_total']].describe()

,cantidad,monto_total
count,24946.000000,24996.000000
mean,1.505131,386.111854
std,0.499984,255.681015
min,1.000000,5.240000
25%,1.000000,180.772500
50%,2.000000,341.925000
75%,2.000000,518.595000
max,2.000000,977.941500


In [134]:
orders_work.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24996 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24996 non-null  object        
 1   id_usuario          24996 non-null  object        
 2   fecha               24996 non-null  datetime64[ns]
 3   pais                24700 non-null  object        
 4   dispositivo         24976 non-null  object        
 5   canal               24966 non-null  object        
 6   nombre_producto     24966 non-null  object        
 7   categoria_producto  24916 non-null  object        
 8   cantidad            24946 non-null  float64       
 9   precio_unitario     24946 non-null  float64       
 10  monto_descuento     24946 non-null  float64       
 11  monto_total         24996 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [135]:
# Vemos filas donde cantidad tiene valores vacíos
orders_work[orders_work['cantidad'].isna()]

,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35


- Como podemos observar, en las filas donde cantidad tiene valores vacíos, también tenemos valores vacíos en precio_unitario y monto_descuento.
- Solo la columna monto_total tiene valores.
- Vamos a imputar valores vacíos en cantidad, precio_unitario y monto descuento de acuerdo al valor de la columna monto total

In [136]:
# vemos cuantos valores vacíos hay en columnas numéricas dataset orders_work (cantidad, precio_unitario, monto_descuento, monto_total)
columnas_numericas = ['cantidad', 'precio_unitario',
       'monto_descuento', 'monto_total']
orders_work[columnas_numericas].isnull().sum()

cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
dtype: int64

In [137]:
# Distribución de valores faltantes de cantidad
# Vamos a ver si los valores vacíos de cantidad siguen un patrón al azar o no
def analizar_valores_faltantes_cantidad(dataframe, columnas_agrupar):
    resultados = {}
    for columna in columnas_agrupar:
        print(f"\n=== Valores faltantes de 'cantidad' agrupados por '{columna}' ===")
        resultado = dataframe['cantidad'].isna().groupby(dataframe[columna]).mean().sort_values(ascending=False)
        print(resultado)
        resultados[columna] = resultado

    return resultados


# Ejecutamos la función
columnas_a_analizar = ['pais', 'dispositivo','canal','nombre_producto','categoria_producto']
resultado = analizar_valores_faltantes_cantidad(orders_work, columnas_a_analizar)



=== Valores faltantes de 'cantidad' agrupados por 'pais' ===
pais
Argentina    0.002755
Colombia     0.001871
Mexico       0.001738
argentina    0.001256
colombia     0.001215
mexico       0.001159
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'dispositivo' ===
dispositivo
desktop    0.002046
mobile     0.001956
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'canal' ===
canal
organic        0.002172
paid_search    0.001933
social         0.001905
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'nombre_producto' ===
nombre_producto
Sneakers-Urban-42       0.004581
Jacket-Winter-M         0.002395
Tablet-Standard-64GB    0.001806
Laptop-Gaming-16GB      0.001438
Vacuum-Pro-Black        0.001437
Phone-Pro-128GB         0.001096
Blender-XL-Red          0.000718
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'categoria_producto' ===
categoria_producto


Conclusión:
- En el caso de dispositivo, canal y nombre_producto los valores faltantes de cantidad son completamente al azar (MCAR = Missing Completely At Random). Esto es, la probabilidad de que falte cantidad es la misma para todos los dispositivos, canales o nombre_producto.
- En el caso de pais, vamos a estandarizar nombre de los paises para calcular la estadística.
- En el caso de categoría_producto los valores faltantes siguen un patrón específico (Missing Not At Random - MNAR). Vamos a imputar los valores de categoría_producto según la información de dataset catalog_work

In [138]:
catalog_work.head(7)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [139]:
# cambiamos Electrónica por Electronica en catalog_work para estandarizar nombre
catalog_work['categoria_producto'] = catalog_work['categoria_producto'].replace({'Electrónica':'Electronica'})
catalog_work.head(7)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electronica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electronica,10.12,King Ltd
2,Tablet-Standard-64GB,Electronica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [140]:
# Unimos orders_work con catalog_work con left join para mantener todas las filas de orders_work y agregar información del catalogo
orders_con_catalog = pd.merge(orders_work, catalog_work, 
                             on='nombre_producto', 
                             how='left')

In [141]:
# verificamos la unión
orders_con_catalog.head()

,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,categoria_producto_x,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,Electronica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,Electronica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,Hogar,176.64,Long-Reid


In [142]:
# vemos información del nuevo dataset
orders_con_catalog.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24996 entries, 0 to 24995
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_pedido             24996 non-null  object        
 1   id_usuario            24996 non-null  object        
 2   fecha                 24996 non-null  datetime64[ns]
 3   pais                  24700 non-null  object        
 4   dispositivo           24976 non-null  object        
 5   canal                 24966 non-null  object        
 6   nombre_producto       24966 non-null  object        
 7   categoria_producto_x  24916 non-null  object        
 8   cantidad              24946 non-null  float64       
 9   precio_unitario       24946 non-null  float64       
 10  monto_descuento       24946 non-null  float64       
 11  monto_total           24996 non-null  float64       
 12  categoria_producto_y  24966 non-null  object        
 13  costo_unitario  

In [143]:
# en categoria_producto_y quedaron 30 productos sin categoria, porque hay 30 valores faltantes en nombre_producto
# eliminamos columnas categoria_producto_x, costo_unitario y proveedor para quedar con dataset orders_work
# Eliminar las columnas especificadas después del left join
orders_work = orders_con_catalog.drop(columns=['categoria_producto_x', 'costo_unitario', 'proveedor'])

In [144]:
# verificamos el cambio
orders_work.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24996 entries, 0 to 24995
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_pedido             24996 non-null  object        
 1   id_usuario            24996 non-null  object        
 2   fecha                 24996 non-null  datetime64[ns]
 3   pais                  24700 non-null  object        
 4   dispositivo           24976 non-null  object        
 5   canal                 24966 non-null  object        
 6   nombre_producto       24966 non-null  object        
 7   cantidad              24946 non-null  float64       
 8   precio_unitario       24946 non-null  float64       
 9   monto_descuento       24946 non-null  float64       
 10  monto_total           24996 non-null  float64       
 11  categoria_producto_y  24966 non-null  object        
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [145]:
# cambiamos nombre de columna categoria_producto_y por categoria_producto
orders_work = orders_work.rename(columns={'categoria_producto_y': 'categoria_producto'})

In [146]:
# Cambiamos argentina por Argentina, mexico por Mexico y colombia por Colombia en columna pais dataset orders
orders_work['pais']=orders_work['pais'].replace({'argentina':'Argentina','mexico':'Mexico','colombia':'Colombia'})

In [147]:
# verificamos el cambio
orders_work['pais'].value_counts()

Mexico       8341
Colombia     8304
Argentina    8055
Name: pais, dtype: int64

In [148]:
# volvemos a ver la distribución de valores faltantes de cantidad en orders_work
# Ejecutamos la función
columnas_a_analizar = ['pais', 'dispositivo','canal','nombre_producto','categoria_producto']
resultado = analizar_valores_faltantes_cantidad(orders_work, columnas_a_analizar)


=== Valores faltantes de 'cantidad' agrupados por 'pais' ===
pais
Argentina    0.002607
Colombia     0.001806
Mexico       0.001678
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'dispositivo' ===
dispositivo
desktop    0.002046
mobile     0.001956
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'canal' ===
canal
organic        0.002172
paid_search    0.001933
social         0.001905
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'nombre_producto' ===
nombre_producto
Sneakers-Urban-42       0.004581
Jacket-Winter-M         0.002395
Tablet-Standard-64GB    0.001806
Laptop-Gaming-16GB      0.001438
Vacuum-Pro-Black        0.001437
Phone-Pro-128GB         0.001096
Blender-XL-Red          0.000718
Name: cantidad, dtype: float64

=== Valores faltantes de 'cantidad' agrupados por 'categoria_producto' ===
categoria_producto
Moda           0.003484
Electronica    0.001448
Hogar          0.0

Conclusión:
- En el caso de país, dispositivo, canal, nombre_producto y categoria_producto los valores faltantes de cantidad son completamente al azar (MCAR = Missing Completely At Random). Esto es, la probabilidad de que falte cantidad es la misma para todos los paises, dispositivos, canal, nombre_producto o categoria_producto. Dichos valores faltantes por lo tanto quedarán como "desconocido". Desconocido es el estándar cuando los valores faltan completamente al azar (Missing Completely At Random)
- Vamos a imputar los 50 valores faltantes a columna cantidad con la mediana para no perder esos datos.
- Vamos a imputar los 50 valores faltantes de monto_descuento con cero, para mantener el monto total.
- Vamos a calcular los valores faltantes de precio_unitario con la fórmula precio_unitario = monto_total / cantidad
- Vamos a crear una columna flag para las 50 filas de valores faltantes de cantidad y luego imputaremos con la mediana.
- La mediana solo se puede imputar si los valores vacíos presentan un patrón MCAR.
- Utilizamos mediana porque:  
  a) no se ve afectada por valores extremos  
  b) mantiene la distribución general de cantidad  
  c) los datos faltan al azar  

In [149]:
# creamos columna flag para valores faltantes de columna cantidad, precio_unitario y monto_descuento
orders_work["cantidad_missing_flag"] = orders_work["cantidad"].isna().astype(int)

In [150]:
# verificamos la nueva columna
orders_work.head()

,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto,cantidad_missing_flag
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,2.0,332.69,0.0,665.37,Moda,0
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,1.0,176.86,5.0,171.86,Electronica,0
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,2.0,102.99,10.0,195.99,Hogar,0
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,1.0,257.87,15.0,242.87,Electronica,0
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,1.0,336.28,0.0,336.28,Hogar,0


In [151]:
# calculamos la mediana de cantidad
mediana_cantidad = orders_work['cantidad'].median()
print(mediana_cantidad)

2.0


In [152]:
# usamos la mediana para imputar los valores faltantes
orders_work['cantidad'] = orders_work['cantidad'].fillna(mediana_cantidad)

In [153]:
# verificamos que la imputación funcionó correctamente
orders_work['cantidad'].isna().sum()

0

In [154]:
# imputamos los 50 valores de monto descuento con cero
orders_work['monto_descuento'] = orders_work['monto_descuento'].fillna(0)

In [155]:
# verificamos que la imputación funcionó correctamente
orders_work['monto_descuento'].isna().sum()

0

In [156]:
# vamos a imputar los 50 valores faltantes de precio_unitario con la fórmula precio_unitario = monto_total / cantidad
orders_work['precio_unitario'] = orders_work['precio_unitario'].fillna(orders_work['monto_total']/orders_work['cantidad'])

In [157]:
# verificamos que la imputación funcionó correctamente
orders_work['precio_unitario'].isna().sum()

0

In [158]:
# volvemos a revisar información de orders_work
orders_work.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24996 entries, 0 to 24995
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_pedido              24996 non-null  object        
 1   id_usuario             24996 non-null  object        
 2   fecha                  24996 non-null  datetime64[ns]
 3   pais                   24700 non-null  object        
 4   dispositivo            24976 non-null  object        
 5   canal                  24966 non-null  object        
 6   nombre_producto        24966 non-null  object        
 7   cantidad               24996 non-null  float64       
 8   precio_unitario        24996 non-null  float64       
 9   monto_descuento        24996 non-null  float64       
 10  monto_total            24996 non-null  float64       
 11  categoria_producto     24966 non-null  object        
 12  cantidad_missing_flag  24996 non-null  int64         
dtypes

Dataset orders_work quedó sin valores vacíos en columnas cantidad, precio_unitario, monto_descuento y monto_total

In [159]:

# después de la limpieza, volvemos a verificar la cantidad de valores vacíos en cada dataset
print('Dataset orders_work')
orders_work.info()
print('Dataset catalog_work')
catalog_work.info()
print('Dataset marketing_work')
marketing_work.info()


Dataset orders_work
<class 'pandas.core.frame.DataFrame'>
Int64Index: 24996 entries, 0 to 24995
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_pedido              24996 non-null  object        
 1   id_usuario             24996 non-null  object        
 2   fecha                  24996 non-null  datetime64[ns]
 3   pais                   24700 non-null  object        
 4   dispositivo            24976 non-null  object        
 5   canal                  24966 non-null  object        
 6   nombre_producto        24966 non-null  object        
 7   cantidad               24996 non-null  float64       
 8   precio_unitario        24996 non-null  float64       
 9   monto_descuento        24996 non-null  float64       
 10  monto_total            24996 non-null  float64       
 11  categoria_producto     24966 non-null  object        
 12  cantidad_missing_flag  24996 non-null  i

In [160]:
# calculamos el porcentaje de valores vacíos de la tabla orders_work

porcentaje_vacios_orders_work = orders_work.isnull().mean()*100
print(porcentaje_vacios_orders_work)


id_pedido                0.000000
id_usuario               0.000000
fecha                    0.000000
pais                     1.184189
dispositivo              0.080013
canal                    0.120019
nombre_producto          0.120019
cantidad                 0.000000
precio_unitario          0.000000
monto_descuento          0.000000
monto_total              0.000000
categoria_producto       0.120019
cantidad_missing_flag    0.000000
dtype: float64


De acuerdo a los datos que tenemos, no se pueden imputar con exactitud los valores nulos de:  
a) país (1,18%,)  
b) dispositivo (0,08%)  
c) canal (0,12%)  
d) nombre de producto (0,12%).  
e) categoria_producto (0,12%)  
- Como habíamos comentado anteriormente, los valores faltantes de estas columnas respecto a las columnas numéricas son completamente al azar (MCAR = Missing Completely At Random).
- Esto es, la probabilidad de que falte un valor numérico es la misma para todos los paises, dispositivos, canal, nombre_producto o categoria_producto.
- Dichos valores faltantes por lo tanto quedarán como "desconocido".
- Desconocido es el estándar cuando los valores faltan completamente al azar (Missing Completely At Random)

In [161]:
# Imputar "desconocido" en columnas señaladas:
columnas_a_imputar = ['pais', 'dispositivo', 'canal', 'nombre_producto', 'categoria_producto']
orders_work[columnas_a_imputar] = orders_work[columnas_a_imputar].fillna("desconocido")

In [162]:
# verificamos la imputación
orders_work.isnull().mean()*100

id_pedido                0.0
id_usuario               0.0
fecha                    0.0
pais                     0.0
dispositivo              0.0
canal                    0.0
nombre_producto          0.0
cantidad                 0.0
precio_unitario          0.0
monto_descuento          0.0
monto_total              0.0
categoria_producto       0.0
cantidad_missing_flag    0.0
dtype: float64

In [163]:
# De acuerdo a información de marketing.info() vemos que faltan canales en dataset
marketing_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [164]:
# Calculamos cuantos canales vacios hay por cada id_campaña
# Creamos una columna auxiliar primero
marketing_work['canal_es_null'] = marketing_work['canal'].isnull()
marketing_work.groupby('id_campaña')['canal_es_null'].sum()

id_campaña
organic_Argentina        11
organic_Colombia         11
organic_Mexico           12
paid_search_Argentina    11
paid_search_Colombia     11
paid_search_Mexico       11
social_Argentina         12
social_Colombia          11
social_Mexico            11
Name: canal_es_null, dtype: int64

In [165]:
# Vamos a imputar el canal según id_campaña
# Creamos mapeo de id_campaña a canal
mapeo_canal = {
    'organic_Argentina': 'organic',
    'organic_Colombia': 'organic', 
    'organic_Mexico': 'organic',
    'paid_search_Argentina': 'paid_search',
    'paid_search_Colombia': 'paid_search',
    'paid_search_Mexico': 'paid_search',
    'social_Argentina': 'social',
    'social_Colombia': 'social',
    'social_Mexico': 'social'
}

# Imputamos valores nulos en canal usando el mapeo
marketing_work['canal'] = marketing_work['canal'].fillna(marketing_work['id_campaña'].map(mapeo_canal))

# Verificamos imputación
marketing_work.isnull().mean()*100

fecha            0.0
pais             0.0
id_campaña       0.0
canal            0.0
gasto            0.0
canal_es_null    0.0
dtype: float64

Columna canal de dataset marketing quedó sin valores faltantes

In [166]:
# después de la limpieza, volvemos a verificar la cantidad de valores vacíos en cada dataset

print('Dataset orders_work')
print(orders_work.isnull().mean())
print('\nDataset catalog_work')
print(catalog_work.isnull().mean())
print('\nDataset marketing_work')
print(marketing_work.isnull().mean())


Dataset orders_work
id_pedido                0.0
id_usuario               0.0
fecha                    0.0
pais                     0.0
dispositivo              0.0
canal                    0.0
nombre_producto          0.0
cantidad                 0.0
precio_unitario          0.0
monto_descuento          0.0
monto_total              0.0
categoria_producto       0.0
cantidad_missing_flag    0.0
dtype: float64

Dataset catalog_work
nombre_producto       0.0
categoria_producto    0.0
costo_unitario        0.0
proveedor             0.0
dtype: float64

Dataset marketing_work
fecha            0.0
pais             0.0
id_campaña       0.0
canal            0.0
gasto            0.0
canal_es_null    0.0
dtype: float64


In [167]:
# vemos con cuantos registro quedó cada dataset
print('Dataset orders_work')
print(orders_work.shape)
print('\nDataset catalog_work')
print(catalog_work.shape)
print('\nDataset marketing_work')
print(marketing_work.shape)

Dataset orders_work
(24996, 13)

Dataset catalog_work
(7, 4)

Dataset marketing_work
(1620, 6)


- Los tres dataset quedaron sin valores nulos.
- Dataset orders_work con 24.996 registros.
- Dataset catalog_work con 7 registros.
- Dataset marketing_work con 1620 registros.

**Resumen de limpieza de datasets:**
- Creamos copia de los datasets
- Validamos y convertimos fechas al formato correcto  
- Revisamos variables numéricas (sin negativos, ceros inválidos, nulos)  
- Eliminamos duplicados  
- Verificamos consistencia de montos  
- Revisamos variables categóricas
- Convertimos a número variables numéricas
- Imputamos categoría en dataset orders de acuerdo a información dataset catalog
- Imputamos desconocido en variables categóricas sin datos.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [168]:
# exportar datasets
orders_work.to_csv('orders_clean.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
catalog_work.to_csv('catalog_clean.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
marketing_work.to_csv('marketing_clean.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

In [169]:
orders_work.head()

,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto,cantidad_missing_flag
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,2.0,332.69,0.0,665.37,Moda,0
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,1.0,176.86,5.0,171.86,Electronica,0
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,2.0,102.99,10.0,195.99,Hogar,0
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,1.0,257.87,15.0,242.87,Electronica,0
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,1.0,336.28,0.0,336.28,Hogar,0


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [178]:
# Cargamos los archivos CSV 
orders_clean = pd.read_csv('orders_clean.csv', sep=';', decimal=',')
catalog_clean = pd.read_csv('catalog_clean.csv', sep=';', decimal=',')
marketing_clean = pd.read_csv('marketing_clean.csv', sep=';', decimal=',')

In [180]:
# calculamos el ingreso total con dataset orders_clean
ingreso_total = orders_clean['monto_total'].sum()
print(f'Ingreso Total: ${ingreso_total:,.2f}')

Ingreso Total: $9,651,251.91


In [181]:
orders_clean.head()


,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto,cantidad_missing_flag
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,2.0,332.69,0.0,665.37,Moda,0
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,1.0,176.86,5.0,171.86,Electronica,0
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,2.0,102.99,10.0,195.99,Hogar,0
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,1.0,257.87,15.0,242.87,Electronica,0
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,1.0,336.28,0.0,336.28,Hogar,0


In [182]:
# paraa calcular el costo total, unimos las tablas orders con catalog con left join
merged_left = pd.merge(orders_clean,catalog_clean, on=['nombre_producto'],how='left')
merged_left.head()

,id_pedido,id_usuario,fecha,pais,dispositivo,canal,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto_x,cantidad_missing_flag,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,2.0,332.69,0.0,665.37,Moda,0,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,1.0,176.86,5.0,171.86,Electronica,0,Electronica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,2.0,102.99,10.0,195.99,Hogar,0,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,1.0,257.87,15.0,242.87,Electronica,0,Electronica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,1.0,336.28,0.0,336.28,Hogar,0,Hogar,176.64,Long-Reid


In [183]:
# calculamos el costo total con la formula suma(costo_unitario x cantidad)
costo_total = (merged_left['costo_unitario']*merged_left['cantidad']).sum()
print(f'Costo Total: ${costo_total:,.2f}')

Costo Total: $3,842,740.09


In [184]:
# cuanto se ha invertido en marketing
marketing_clean.head()

,fecha,pais,id_campaña,canal,gasto,canal_es_null
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25,False
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34,False
2,2025-01-01,Mexico,social_Mexico,social,2045.01,False
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21,False
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40,False


In [185]:
# cuanto se ha invertido en marketing
costo_marketing = marketing_clean['gasto'].sum()
print(f'Costo Marketing: ${costo_marketing:,.2f}')

Costo Marketing: $2,871,843.53


In [186]:
# calculamos el profit
profit = ingreso_total - costo_total - costo_marketing
print(f'Profit: ${profit:,.2f}')

Profit: $2,936,668.29


In [187]:
# ticket promedio
ticket_promedio = merged_left['monto_total'].mean()
print(f'Ticket Promedio: ${ticket_promedio:,.2f}')

Ticket Promedio: $386.11


In [188]:
# cantidad promedio de productos por orden
cant_prom_productos_por_orden = merged_left['cantidad'].mean()
print(f'Cantidad promedio de productos por orden: {cant_prom_productos_por_orden:,.2f}')

Cantidad promedio de productos por orden: 1.51


In [189]:
# producto más vendido
merged_left.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

nombre_producto
Vacuum-Pro-Black        6296.0
Blender-XL-Red          6285.0
Jacket-Winter-M         6276.0
Sneakers-Urban-42       6210.0
Laptop-Gaming-16GB      4226.0
Tablet-Standard-64GB    4163.0
Phone-Pro-128GB         4146.0
desconocido               45.0
Name: cantidad, dtype: float64

El producto más vendido es la aspiradora Vacuum-Pro-Black con 6.296 unidades vendidas

In [190]:
# cuanto se ha invertido en marketing por canal
marketing_clean.groupby('canal')['gasto'].sum().sort_values(ascending=False)

canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [191]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [192]:
# Explorar tabla events

# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()


,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [193]:
# Nombre de todos los eventos
# =========================
query_events = '''
SELECT
DISTINCT(nombre_evento)
FROM events
ORDER BY(nombre_evento);
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,nombre_evento
0,add_payment_info
1,add_to_cart
2,begin_checkout
3,first_visit
4,purchase


In [194]:
# PARTE 1: Totales del funnel 
# Calcularemos cuantos usuarios llegan a cada etapa del funnel con CTEs (Common Table Expressions)
# Las CTEs son subconsultas con nombres que organizan las querys en una secuencia lógica (WITH nombre_cte AS)
# Sirve para nombrar cada etapa del funnel como registro, carrito, compra, etc.
# Se pueden organizar en un orden correcto usando time_stamp.

# ======================



query_totals = '''
WITH
first_visit_cte AS(SELECT DISTINCT id_usuario FROM events WHERE nombre_evento='first_visit'),
add_to_cart_cte AS(SELECT DISTINCT id_usuario FROM events WHERE nombre_evento='add_to_cart'),
begin_checkout_cte AS(SELECT DISTINCT id_usuario FROM events WHERE nombre_evento='begin_checkout'),
add_payment_info_cte AS(SELECT DISTINCT id_usuario FROM events WHERE nombre_evento='add_payment_info'),
purchase_cte AS(SELECT DISTINCT id_usuario FROM events WHERE nombre_evento='purchase')
SELECT
(SELECT COUNT (*) FROM first_visit_cte) AS first_visit,
(SELECT COUNT (*) FROM add_to_cart_cte) AS add_to_cart,
(SELECT COUNT (*) FROM begin_checkout_cte) AS begin_checkout,
(SELECT COUNT (*) FROM add_payment_info_cte) AS add_payment_info,
(SELECT COUNT (*) FROM purchase_cte) AS purchase;
'''

totals = pd.read_sql(query_totals, con=engine)
totals


,first_visit,add_to_cart,begin_checkout,add_payment_info,purchase
0,7796,7634,7208,6250,6240


In [195]:
# PARTE 2: Tasa de conversiones entre cada etapa del funnel
# ======================



query_conversion = '''
WITH first_visit AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento='first_visit'
),
add_to_cart AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento='add_to_cart'
),
begin_checkout AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento='begin_checkout'
),
add_payment_info AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento='add_payment_info'
),
purchase AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento='purchase'
)

SELECT
ROUND(((SELECT COUNT(*) FROM add_to_cart)*100.0/NULLIF((SELECT COUNT(*) FROM first_visit),0)),2) AS conversion_add_to_cart,
ROUND(((SELECT COUNT(*) FROM begin_checkout)*100.0/NULLIF((SELECT COUNT(*) FROM first_visit),0)),2) AS conversion_begin_checkout,
ROUND(((SELECT COUNT(*) FROM add_payment_info)*100.0/NULLIF((SELECT COUNT(*) FROM first_visit),0)),2) AS conversion_add_payment_info,
ROUND(((SELECT COUNT(*) FROM purchase)*100.0/NULLIF((SELECT COUNT(*) FROM first_visit),0)),2) AS conversion_purchase


'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase
0,97.92,92.46,80.17,80.04


La tasa de conversión entre cada etapa del funnel es:
- Add to cart: 97,92% (de cada 100 usuarios que visitan la página, 97 llegan a add_to_cart)
- Begin checkout: 92,46% (de cada 100 usuarios que visitan la página, 92 pasan a begin checkout)
- Add payment info: 80,17% (de cada 100 usuarios que visitan la página, 80 llegan a add payment info)
- Purchase: 80,04% (de cada 100 usuarios que visitan la página, 80 llegan a purchase)
- La mayor cantidad de usuarios se pierden en la etapa de begin_chekout. De cada 100 usuarios que visitan la página, 92 llegan a begin checkout y de esos 92, solo 80 llegan a add_payment_info

---

- La tasa de conversión final es de 80,04%
- Esto quiere decir que por cada 100 usuarios que visitan la página por primera vez, 80 usuarios compran.

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [196]:
# Explorar tabla users
# =========================

query_users = '''
SELECT *
FROM users;
'''

users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [197]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


---

In [198]:
# Retención por cohortes
# Se identifica la cohorte de cada usuario según el mes de registro
# ======================

query_cohort_retention_final = '''

WITH cohortes AS (
    SELECT 
        -- Paso 1: seleccionamos la columna id_usuario de la tabla users
        id_usuario, 
        -- Paso 2: asigna la cohorte a cada id_usuario de la tabla users
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohort_mes 
    FROM users
    GROUP BY id_usuario, cohort_mes
)

SELECT *
FROM cohortes

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,id_usuario,cohort_mes
0,user_2207,2025-04-01 00:00:00+00:00
1,user_105,2025-02-01 00:00:00+00:00
2,user_7782,2025-01-01 00:00:00+00:00
3,user_6826,2025-01-01 00:00:00+00:00
4,user_3760,2025-02-01 00:00:00+00:00
...,...,...
7995,user_93,2025-03-01 00:00:00+00:00
7996,user_2859,2025-02-01 00:00:00+00:00
7997,user_4574,2025-01-01 00:00:00+00:00
7998,user_557,2025-01-01 00:00:00+00:00


In [199]:
# Retención por cohortes
# Se calcula la retención semanal: cuántos usuarios se mantienen activos en cada semana desde su registro.
# ======================
query_cohort_retention_final = '''
WITH cohortes AS ( 
    -- Paso 1: seleccionamos la columna id_usuario de la tabla users
    SELECT 
        ua.id_usuario,
        DATE_TRUNC('month',CAST(u.fecha_registro AS DATE)) AS cohort_mes,
        ua.dias_despues_registro,
        ua.activo
    FROM user_activity AS ua
    LEFT JOIN users as u
    ON u.id_usuario = ua.id_usuario
    GROUP BY ua.id_usuario, u.fecha_registro, ua.dias_despues_registro, ua.activo
)

SELECT
cohort_mes,
COUNT(*) AS clientes_iniciales,
COUNT (CASE WHEN dias_despues_registro >=7 AND activo= 1 THEN 1 END) AS retenido_w1,
COUNT (CASE WHEN dias_despues_registro >=14 AND activo= 1 THEN 1 END) AS retenido_w2,
COUNT (CASE WHEN dias_despues_registro >=21 AND activo= 1 THEN 1 END) AS retenido_w3
FROM cohortes
GROUP BY cohort_mes

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final




,cohort_mes,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3
0,2025-02-01 00:00:00+00:00,5776,2430,1819,1210
1,2025-04-01 00:00:00+00:00,6424,2692,2012,1315
2,2025-03-01 00:00:00+00:00,6544,2745,2068,1363
3,2025-01-01 00:00:00+00:00,6508,2692,1995,1327
4,2025-05-01 00:00:00+00:00,6748,2756,2061,1385


In [200]:
# Retención por cohortes
# Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:
# ======================
query_cohort_retention_final = '''
WITH cohortes AS (
    -- Paso 1: Identificar cohortes
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mes
    FROM users
), actividad AS (
    -- Paso 2: Calcular las semanas de actividad
    SELECT
        id_usuario,
        FLOOR(dias_despues_registro / 7.0) AS semana_num,
        activo
    FROM user_activity
), base AS (
    -- Paso 3: Unir cohorte con actividad
    SELECT 
        c.cohorte_mes,
        c.id_usuario,
        act.semana_num,
        act.activo
    FROM cohortes AS c
    LEFT JOIN actividad AS act
        ON c.id_usuario = act.id_usuario
), retencion AS (
    -- Paso 4: Calcular retencion semanal
    SELECT
        cohorte_mes,
        COUNT(DISTINCT id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE
            WHEN semana_num >=1 AND activo = 1 THEN id_usuario
            END) AS retenidos_w1,
        COUNT(DISTINCT CASE
            WHEN semana_num >=2 AND activo = 1 THEN id_usuario
            END) AS retenidos_w2,
        COUNT(DISTINCT CASE
            WHEN semana_num >=3 AND activo = 1 THEN id_usuario
            END) AS retenidos_w3
    FROM base
    GROUP BY cohorte_mes
)
SELECT
    TO_CHAR(cohorte_mes, 'YYYY-MM') AS cohorte,
    clientes_iniciales,
    retenidos_w1,
    retenidos_w2,
    retenidos_w3,
    ROUND(1.0*retenidos_w1 / clientes_iniciales, 2) AS semana_1,
    ROUND(1.0*retenidos_w2 / clientes_iniciales, 2) AS semana_2,
    ROUND(1.0*retenidos_w3 / clientes_iniciales, 2) AS semana_3
FROM retencion
ORDER BY cohorte
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final




,cohorte,clientes_iniciales,retenidos_w1,retenidos_w2,retenidos_w3,semana_1,semana_2,semana_3
0,2025-01,1627,1381,1253,1027,0.85,0.77,0.63
1,2025-02,1444,1255,1154,940,0.87,0.80,0.65
2,2025-03,1636,1428,1306,1060,0.87,0.80,0.65
3,2025-04,1606,1394,1261,1022,0.87,0.79,0.64
4,2025-05,1687,1446,1321,1088,0.86,0.78,0.64


## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - 
**H₀ (Hipótesis nula):** ...
**H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...


In [201]:
# Cargarlibrerias
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

In [202]:
# Cargar dataset y verificar la carga
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
df.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [203]:
# Verificamos que no existan valores vacíos en columnas críticas
df.isna().sum()


id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64

No existen valores vacíos

In [204]:
# Vemos información estadística del dataset para verificar que no existen outliers
df.describe()

,convirtio,duracion_sesion
count,10000.000000,10000.000000
mean,0.159900,159.862439
std,0.366532,81.074410
min,0.000000,20.010000
25%,0.000000,89.470000
50%,0.000000,159.725000
75%,0.000000,229.745000
max,1.000000,300.000000


Tenemos 10.000 registros (entre 0 y 1)

Planteamos las hipótesis:  
**H₀ (Hipótesis nula):** no hay diferencias entre la situación actual y el tratamiento.  
**H₁ (Hipótesis alternativa):** la nueva interfaz del usuario en el checkout mejora los resultados de conversión. Plantea que sí existe una diferencia
   
**Test estadístico:** prueba z de proporciones (z-test).  
Pregunta de negocio: ¿la proporción es diferente entre el grupo control y el grupo de prueba? 

**Nivel de significancia alpha:** 0.05 (umbral de significancia)

# Totales

In [206]:
con_ct = df.groupby('variante')['convirtio'].sum()
con_ct

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

In [207]:
total = df.groupby('variante')['convirtio'].count()
total

variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [209]:
exitos = con_ct.values
observaciones = total.values

In [210]:
# Calculo de la prueba z. Aplicamos prueba y visualizamos resultados.
z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadístico z:{z_stat}")
print(f"Valor p:{p_value}")

Estadístico z:-0.8132782986429474
Valor p:0.41605851639119995


In [211]:
# Interpretación de la prueba
alpha = 0.05 # nivel de significancia
if p_value < alpha:
    print("Rechazamos la hipótesis nula. Hay evidencia de una diferencia")
else:
    print("No rechazamos la hipótesis nula. No hay evidencia suficiente de una diferencia")

No rechazamos la hipótesis nula. No hay evidencia suficiente de una diferencia


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

# 6.1 Resumen ejecutivo

**Hallazgos clave**

- El negocio está creciendo de forma estable y rentable, aunque existe dependencia de ciertos productos líderes y una caída puntual en febrero que conviene investigar.  
- Las ventas son altas (\$9,65 MM) con crecimiento sostenido.  
- Las ventas YTD suben de forma constante.  
- Después de la caída de Febrero, las ventas se recuperan y vuelven al promedio.  
- El Profit acompaña con \$5,8 MM sobre $9,65 MM vendidos. Esto sugiere márgenes saludables.
- Vacuum-Pro-Black y Sneakers-Urban-42 lideran claramente, ambos de la linea Hogar.
- Laptop Gaming 16G con pérdidas.
- La caída de Febrero es una anomalía que conviene investigar (estacionalidad, menos dias, quiebre de stock, vacaciones, marketing débil)

**Métricas principales**

- Ventas Totales: \$9.651.252
- Profit: \$5.808.512
- Ticket promedio: \$386
- Productos por pedido: 1,51

**Insight accionables**

- Las ventas dependen de pocos productos líderes → riesgo de concentración, por lo que se recomienda diversificar catálogo y potenciar productos secundarios.
- Laptop Gaming 16G vende pero pierde dinero. Revisar urgente: pricing, descuentos, costos, garantía, logística.
- Probar aumento gradual de precio en Laptop Gaming → medir impacto en margen y demanda. Si el volumen acompaña probar +3%, luego +5%.
- Crear bundles: laptop + mouse, laptop + mochila, laptop + monitor para subir ticket promedio y mejorar la rentabilidad.
- Si laptop pierde dinero analizar: ¿atrae clientes que luego compran otras cosas?¿o solo destruye margen?
- El ticket promedio bajo (\$386) indica una oportuidad de cross-selling, upselling, promociones por carrito mínimo.
- Priorizar productos con alta venta + alto margen (Vacuum-Pro-Black y Sneakers-Urban-42): priorizar stock, proteger margen, invertir más en marketing.
- Phone-Pro-128GB y Tablet-Standard-64GB son productos sanos, con buen equilibrio entre ventas y rentabilidad.Mantener pricing, empujarlos como productos core.
- Blender-XL-Red y Jacket-Winter-M venden mucho pero con margen débil. Posibles descuentos excesivos/costos altos. Revisar estructura de costos, subir precio, reducir promociones.
- "Desconocido” indica problema de calidad de datos. Productos sin clasificación. Corregir maestro de productos, validar ETL/carga Power BI.
- La empresa debe segmentar productos en 4 cuadrantes:
  | Tipo | Acción |
  |------|-------| 
  |Alta venta + alto profit | Escalar |
  |Alta venta + bajo profit | Optimizar margen|
  |Baja venta + alto profit | Potencias marketing |
  |Baja venta + bajo profit | Eliminar|


## Entrega Final
Se comparte el acceso al proyecto final para revisión.

Se entrega proyecto utilizando Power BI.

Opciones para publicación:

🔗 Link público del dashboard publicado en Power BI Service o Tableau Public / Tableau Cloud  
🔗 Link de Google Drive o OneDrive con el archivo .pbix del proyecto  
✅ Confirmación de que el archivo .pbix fue subido según las instrucciones  
📎 Enlace del proyecto  

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive